# 🏥 Tech Challenge FIAP - Fase 1
## Notebook 03: Modelagem de Machine Learning

---

### 📋 **Objetivos deste Notebook**

1. ✅ Carregar dados pré-processados
2. ✅ Treinar múltiplos modelos de classificação
3. ✅ Comparar desempenho inicial
4. ✅ Salvar modelos treinados
5. ✅ Documentar hiperparâmetros

---

**Input**: Dados pré-processados (X_train, X_test, y_train, y_test)
**Output**: Modelos treinados + Predições + Métricas iniciais

**Modelos a treinar**:
- ✅ Regressão Logística
- ✅ Random Forest
- ✅ Support Vector Machine (SVM)
- ✅ Gradient Boosting (XGBoost)

## 📚 1. Importação de Bibliotecas

In [ ]:
# Manipulação de dados
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# Métricas
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    roc_auc_score
)

# Salvamento
import joblib
import pickle
from datetime import datetime
import time

# Configurações
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
RANDOM_STATE = 42

print("✅ Bibliotecas importadas com sucesso!")
print(f"🎲 Random State: {RANDOM_STATE}")

## 📂 2. Carregamento dos Dados Pré-processados

In [ ]:
# Carregar dados do Notebook 02
print("📥 Carregando dados pré-processados...")

X_train = np.load('../data/processed/X_train.npy')
X_test = np.load('../data/processed/X_test.npy')
y_train = np.load('../data/processed/y_train.npy')
y_test = np.load('../data/processed/y_test.npy')

# Carregar nomes das features
with open('../data/processed/feature_names.pkl', 'rb') as f:
    feature_names = pickle.load(f)

# Carregar informações do split
with open('../data/processed/split_info.pkl', 'rb') as f:
    split_info = pickle.load(f)

print("✅ Dados carregados com sucesso!")
print("\n📊 DIMENSÕES:")
print(f"   X_train: {X_train.shape}")
print(f"   X_test: {X_test.shape}")
print(f"   y_train: {y_train.shape}")
print(f"   y_test: {y_test.shape}")
print(f"\n📋 Features: {len(feature_names)}")

## 🤖 3. Configuração dos Modelos

Vamos treinar **4 modelos diferentes** e compará-los!

In [ ]:
# Dicionário com todos os modelos
models = {
    'Logistic Regression': LogisticRegression(
        random_state=RANDOM_STATE,
        max_iter=1000,
        solver='lbfgs'
    ),
    
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=2,
        n_jobs=-1
    ),
    
    'Support Vector Machine': SVC(
        kernel='rbf',
        C=1.0,
        gamma='scale',
        random_state=RANDOM_STATE,
        probability=True  # Para calcular ROC-AUC
    ),
    
    'XGBoost': XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        random_state=RANDOM_STATE,
        eval_metric='logloss',
        n_jobs=-1
    )
}

print("✅ Modelos configurados:")
for i, name in enumerate(models.keys(), 1):
    print(f"   {i}. {name}")

## 🏋️ 4. Treinamento dos Modelos

In [ ]:
# Dicionário para armazenar modelos treinados
trained_models = {}
training_times = {}

print("🏋️ INICIANDO TREINAMENTO DOS MODELOS")
print("="*80)

for name, model in models.items():
    print(f"\n🔧 Treinando: {name}")
    print(f"   Parâmetros: {model.get_params()}")
    
    # Medir tempo de treinamento
    start_time = time.time()
    
    # Treinar modelo
    model.fit(X_train, y_train)
    
    # Calcular tempo
    elapsed_time = time.time() - start_time
    training_times[name] = elapsed_time
    
    # Salvar modelo treinado
    trained_models[name] = model
    
    print(f"   ✅ Treinado em {elapsed_time:.2f} segundos")

print("\n" + "="*80)
print("✅ TODOS OS MODELOS TREINADOS COM SUCESSO!")
print("="*80)

## 📊 5. Predições

In [ ]:
# Fazer predições para todos os modelos
predictions = {}
predictions_proba = {}

print("🔮 Fazendo predições para todos os modelos...")

for name, model in trained_models.items():
    # Predições de classe
    y_pred = model.predict(X_test)
    predictions[name] = y_pred
    
    # Probabilidades (para ROC-AUC)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    predictions_proba[name] = y_pred_proba
    
    print(f"   ✅ {name}: {len(y_pred)} predições")

print("\n✅ Todas as predições realizadas!")

## 📈 6. Avaliação Inicial dos Modelos

In [ ]:
# Calcular métricas para todos os modelos
results = []

print("📊 AVALIAÇÃO DOS MODELOS")
print("="*100)

for name in trained_models.keys():
    y_pred = predictions[name]
    y_pred_proba = predictions_proba[name]
    
    # Calcular métricas
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    results.append({
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'ROC-AUC': roc_auc,
        'Training Time (s)': training_times[name]
    })
    
    print(f"\n🤖 {name}:")
    print(f"   Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"   Precision: {precision:.4f} ({precision*100:.2f}%)")
    print(f"   Recall:    {recall:.4f} ({recall*100:.2f}%)")
    print(f"   F1-Score:  {f1:.4f} ({f1*100:.2f}%)")
    print(f"   ROC-AUC:   {roc_auc:.4f} ({roc_auc*100:.2f}%)")
    print(f"   Tempo:     {training_times[name]:.2f}s")

print("\n" + "="*100)

In [ ]:
# Criar DataFrame com resultados
results_df = pd.DataFrame(results)
results_df = results_df.round(4)

print("📊 TABELA COMPARATIVA DE RESULTADOS:")
print("="*100)
print(results_df.to_string(index=False))
print("="*100)

# Salvar resultados
results_df.to_csv('../reports/model_comparison.csv', index=False)
print("\n✅ Resultados salvos em: reports/model_comparison.csv")

## 📊 7. Visualização Comparativa

In [ ]:
# Gráfico comparativo de métricas
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(results_df))
width = 0.15

for i, metric in enumerate(metrics_to_plot):
    offset = width * (i - 2)
    ax.bar(x + offset, results_df[metric], width, label=metric, alpha=0.8)

ax.set_xlabel('Modelos', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Comparação de Métricas por Modelo', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(results_df['Model'], rotation=15, ha='right')
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.3)
ax.set_ylim([0.8, 1.0])  # Ajustar escala para melhor visualização

plt.tight_layout()
plt.savefig('../reports/figures/10_model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Gráfico salvo em: reports/figures/10_model_comparison.png")

In [ ]:
# Gráfico de tempo de treinamento
fig, ax = plt.subplots(figsize=(10, 5))

colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']
bars = ax.barh(results_df['Model'], results_df['Training Time (s)'], color=colors, alpha=0.8)

ax.set_xlabel('Tempo de Treinamento (segundos)', fontsize=12, fontweight='bold')
ax.set_title('Comparação de Tempo de Treinamento', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# Adicionar valores nas barras
for i, bar in enumerate(bars):
    width = bar.get_width()
    ax.text(width, bar.get_y() + bar.get_height()/2, 
            f'{width:.2f}s', ha='left', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/11_training_time.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Gráfico salvo em: reports/figures/11_training_time.png")

## 🏆 8. Melhor Modelo

In [ ]:
# Identificar melhor modelo por métrica
print("🏆 RANKING DOS MODELOS")
print("="*80)

for metric in metrics_to_plot:
    best_idx = results_df[metric].idxmax()
    best_model = results_df.loc[best_idx, 'Model']
    best_score = results_df.loc[best_idx, metric]
    print(f"\n🥇 Melhor {metric}: {best_model}")
    print(f"   Score: {best_score:.4f} ({best_score*100:.2f}%)")

print("\n" + "="*80)

# Modelo com melhor F1-Score (balanceado)
best_f1_idx = results_df['F1-Score'].idxmax()
best_overall_model = results_df.loc[best_f1_idx, 'Model']

print(f"\n🎯 MODELO RECOMENDADO (baseado em F1-Score): {best_overall_model}")
print("\n💡 F1-Score é uma boa métrica para datasets balanceados,")
print("   pois considera tanto precisão quanto recall.")

## 💾 9. Salvamento dos Modelos

In [ ]:
# Criar diretório para modelos
import os
os.makedirs('../models/saved_models', exist_ok=True)

print("💾 Salvando todos os modelos treinados...")
print("="*80)

# Salvar cada modelo
for name, model in trained_models.items():
    # Nome do arquivo (sem espaços)
    filename = name.lower().replace(' ', '_')
    filepath = f'../models/saved_models/{filename}.joblib'
    
    # Salvar modelo
    joblib.dump(model, filepath)
    print(f"   ✅ {name}: {filepath}")

print("\n✅ Todos os modelos salvos!")

In [ ]:
# Salvar predições
predictions_dict = {
    'y_test': y_test,
    'predictions': predictions,
    'predictions_proba': predictions_proba
}

with open('../models/predictions.pkl', 'wb') as f:
    pickle.dump(predictions_dict, f)

print("✅ Predições salvas em: models/predictions.pkl")

In [ ]:
# Salvar metadados dos modelos
metadata = {
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'random_state': RANDOM_STATE,
    'models': list(trained_models.keys()),
    'best_model': best_overall_model,
    'training_times': training_times,
    'feature_names': feature_names,
    'n_features': len(feature_names),
    'n_train_samples': len(X_train),
    'n_test_samples': len(X_test)
}

with open('../models/model_metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)

print("✅ Metadados salvos em: models/model_metadata.pkl")

## 📊 10. Resumo da Modelagem

In [ ]:
print("="*80)
print("📊 RESUMO DA MODELAGEM")
print("="*80)

print("\n✅ MODELOS TREINADOS:")
for i, name in enumerate(trained_models.keys(), 1):
    print(f"   {i}. {name}")

print("\n📊 DADOS UTILIZADOS:")
print(f"   • Features: {len(feature_names)}")
print(f"   • Treino: {len(X_train):,} amostras")
print(f"   • Teste: {len(X_test):,} amostras")

print("\n🏆 MELHORES RESULTADOS:")
print(f"   • Melhor Accuracy: {results_df['Accuracy'].max():.4f}")
print(f"   • Melhor F1-Score: {results_df['F1-Score'].max():.4f}")
print(f"   • Melhor ROC-AUC: {results_df['ROC-AUC'].max():.4f}")

print(f"\n🎯 MODELO RECOMENDADO: {best_overall_model}")

print("\n⏱️ TEMPOS DE TREINAMENTO:")
for name, time_val in training_times.items():
    print(f"   • {name}: {time_val:.2f}s")

total_time = sum(training_times.values())
print(f"   • TOTAL: {total_time:.2f}s ({total_time/60:.2f} minutos)")

print("\n💾 ARQUIVOS SALVOS:")
print("   • models/saved_models/*.joblib (4 modelos)")
print("   • models/predictions.pkl")
print("   • models/model_metadata.pkl")
print("   • reports/model_comparison.csv")
print("   • reports/figures/10_model_comparison.png")
print("   • reports/figures/11_training_time.png")

print("\n" + "="*80)

## 📝 11. Observações e Insights

### **🔍 Análise dos Resultados:**

1. **Performance Geral**:
   - Todos os modelos apresentam performance excelente (>90%)
   - Dataset está bem balanceado e separável
   - Normalização foi efetiva

2. **Trade-offs**:
   - **Logistic Regression**: Rápido, interpretável, boa baseline
   - **Random Forest**: Robusto, menos sensível a hiperparâmetros
   - **SVM**: Excelente para dados de alta dimensão
   - **XGBoost**: Alta performance, mas requer mais tuning

3. **Considerações Médicas**:
   - Para diagnóstico de câncer, **Recall** é crítico (minimizar falsos negativos)
   - **Precision** também importante (evitar alarmes falsos)
   - F1-Score balanceia ambos

### **⚠️ IMPORTANTE:**

> Estes modelos são ferramentas de **apoio à decisão médica**.  
> O diagnóstico final SEMPRE deve ser feito por um profissional de saúde.  
> Modelos de ML **complementam**, mas não **substituem** a expertise médica.

---

## ➡️ Próximos Passos

### **No Notebook 04 - Avaliação:**
1. Análise detalhada das métricas
2. Matriz de confusão para cada modelo
3. Curvas ROC e Precision-Recall
4. Análise de erros
5. Comparação estatística

### **No Notebook 05 - Explicabilidade:**
1. SHAP values (OBRIGATÓRIO FIAP)
2. Feature importance
3. Interpretação clínica
4. Casos de uso práticos

---

## ✅ Conclusão do Notebook 03

**Status**: ✅ Completo

**Modelos Treinados**: 4

**Tempo Total**: [Anotar após execução]

**Próximo**: `04_avaliacao.ipynb`